In [1]:
import pandas as pd
# use dev to pull few shot examples
dev = pd.read_csv('val.csv')

# filter dev where 'text' is less than 100 words to not make prompt huge
dev['word_count'] = dev['text'].str.split().apply(len)
filtered_dev = dev[dev['word_count'] < 100]

print("Original size:", len(dev))
print("Filtered size (<100 words):", len(filtered_dev))

human_dev = filtered_dev[filtered_dev['label'] == 0]
ai_dev = filtered_dev[filtered_dev['label'] == 1]

Original size: 84899
Filtered size (<100 words): 23325


In [2]:
print('running')

prompt_template = """You are an expert human vs AI classifier and will determine the source of a text. Output exactly a 0 for human or 1 for AI with no extra explanation or delimiters. Here are a couple human generated examples:
- Input: {human_example_1}
- Output: 0
- Input: {human_example_2}
- Output: 0
Here are a few AI generated examples:
- Input: {ai_example_1}
- Output: 1
- Input: {ai_example_2}
- Output: 1
Here is the text you will classify:
- Input: {text}
- Output: """

test = pd.read_csv('test.csv')

import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
checkpoint_file = "few-shot_llm_results.csv"

# load already processed indices if file exists
done_indices = set()
if os.path.exists(checkpoint_file):
    prev = pd.read_csv(checkpoint_file)
    if "i" in prev.columns:
        done_indices = set(prev["i"].tolist())
        print(f"Resuming: {len(done_indices)} rows already done")

import openai
import time

start = time.time()

class_sci_api_key = 'sk-j49dRg-v03_qWy-1xijenA'

def call_llm(prompt, model='gemma3'):
    """ Make an API call to Pitt SCI LLM 
        Args:
            model: {gemma3, llama3.1, deepseek-r1}
    """
    client = openai.OpenAI( 
        api_key=class_sci_api_key, 
        base_url="https://ol.sci.pitt.edu"
    )
    response = client.chat.completions.create( 
        model = model, # model to send to the proxy 
        messages = [{ 
                "role": "user", 
                "content": prompt
        }] 
    ) 
    response_text = response.choices[0].message.content
    return response_text

results = []
write_lock = threading.Lock()

def process_row(i, row):
    text = row["text"]
    label = row["label"]
    
    # pull 2 random examples of each
    human_examples = human_dev.sample(n=2, random_state=i)["text"].tolist()
    ai_examples = ai_dev.sample(n=2, random_state=i)["text"].tolist()

    prompt = prompt_template.format(
        human_example_1=human_examples[0],
        human_example_2=human_examples[1],
        ai_example_1=ai_examples[0],
        ai_example_2=ai_examples[1],
        text=text
    )
    
    try:
        response = call_llm(prompt, model="gemma3")
        pred_text = str(response).strip()
        if pred_text not in {"0", "1"}:
            pred_text = pred_text[0] if len(pred_text) > 0 and pred_text[0] in {"0", "1"} else None
        
        prediction = int(pred_text) if pred_text is not None else 0

    except Exception as e:
        print(f"iteration {i} failed: {e}")
        prediction = 0  # <-- default on 500 / any failure

    row_dict = {
        "i": i,
        "text": text,
        "label": label,
        "prediction": prediction
    }
    
    return i, row_dict

remaining = [(i, row) for i, row in test.iterrows() if i not in done_indices]

max_workers = 20

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_row, i, row) for i, row in remaining]

    for future in as_completed(futures):
        i, row_dict = future.result()

        print(f'iteration {i} output: {row_dict["prediction"]} label: {row_dict["label"]}')
        
        results.append(row_dict)

        # SAVE IMMEDIATELY (append mode)
        with write_lock:
            pd.DataFrame([row_dict]).to_csv(
                checkpoint_file,
                mode='a',
                header=not os.path.exists(checkpoint_file),
                index=False
            )

end = time.time()
duration = round(end-start,2)
print(f'duration: {duration}s')

running
Resuming: 36833 rows already done
iteration 36850 output: 0 label: 1
iteration 36847 output: 0 label: 1
iteration 36844 output: 0 label: 0
iteration 36848 output: 1 label: 0
iteration 36846 output: 1 label: 1
iteration 36838 output: 1 label: 1
iteration 36845 output: 0 label: 0
iteration 36834 output: 1 label: 1
iteration 36840 output: 0 label: 0
iteration 36851 output: 0 label: 1
iteration 36841 output: 0 label: 0
iteration 36843 output: 0 label: 1
iteration 36852 output: 1 label: 1
iteration 36842 output: 0 label: 1
iteration 36849 output: 0 label: 1
iteration 36837 output: 0 label: 1
iteration 36830 output: 1 label: 0
iteration 36835 output: 1 label: 1
iteration 36836 output: 1 label: 1
iteration 36839 output: 1 label: 0
iteration 36853 output: 1 label: 1
iteration 36856 output: 1 label: 1
iteration 36854 output: 1 label: 0
iteration 36855 output: 1 label: 1
iteration 36857 output: 0 label: 0
iteration 36858 output: 0 label: 1
iteration 36862 output: 1 label: 1
iteration 368